In [25]:
import pyxid2
import time

#dir(pyxid2) # available functions and classes
#help(pyxid2) # for more info on the library

### See if the XidDevice is recognized
- ensure the first switch is set to standard no keyboard mode

In [26]:
device = pyxid2.get_xid_devices()
print(dev) # list of connected devices

dev = device[0] # select the first device

<XidDevice "Riponda Model E">


### Some test code to see if the thing works

In [27]:
dev.reset_timer() # reset the timer to 0 ms

start = time.time() #returns the current time in seconds 

while time.time() - start < 10:
    dev.poll_for_response() # check for responses  
    if dev.has_response(): # do we have responses in the queue
        response = dev.get_next_response() # returns a dictionary with response info
        if response['pressed']:
            print(f"Button {response['key']}  |  RT: {response['time']} ms")

Button 3  |  RT: 2877 ms
Button 4  |  RT: 3458 ms
Button 0  |  RT: 4758 ms
Button 1  |  RT: 5353 ms
Button 2  |  RT: 5600 ms


### Function for the text screens

In [28]:
def move_on():
    dev.flush_serial_buffer() # clear the buffer to avoid getting old responses
     
    while True:
        dev.poll_for_response() # check for new responses  
        if dev.has_response(): # if a response is queued
            response = dev.get_next_response()
            if response['pressed']: # act when pressed, not released
                if response['key'] != 0: # any key except the first one returns space
                    key = 'space'
                else:
                    key = 'escape'

                return key

In [30]:
key = move_on()
print(key)

escape


### Function for trial responses

In [31]:
def trial_resp_pad(images, max_images=10):
    if images <= 1:  # at the start of the trial
        dev.flush_serial_buffer() # clear old responses
        dev.reset_timer() # reset timer to 0 ms at the start of the trial
    
    dev.poll_for_response()  # check device for new responses
    if dev.has_response():   # if a response is queued
        response = dev.get_next_response()
        trial_keys = [0, 3, 4]  # valid response keys: escape, left, right
        if response['pressed'] and response['key'] in trial_keys:  # ignore releases and invalid keys
            resp_time = response['time']  # RT in ms since reset_timer
            if response['key'] == 0:
                return 'escape', resp_time
            elif response['key'] == 3:
                return 'left', resp_time
            elif response['key'] == 4:
                return 'right', resp_time

    if images == max_images:  # no response by end of trial
        resp_time = dev.query_timer()  # get total trial duration
        key = 'none'
        return key, resp_time


In [32]:
images = 0

while images < 10:
    images += 1
    print(f"Image {images}")
    time.sleep(0.25)
    result = trial_resp_pad(images)
    if result is not None:
        key, rt = result
        break

print(f"Key: {key} | RT: {rt} ms")

Image 1
Image 2
Image 3
Image 4
Image 5
Image 6
Image 7
Image 8
Image 9
Key: left | RT: 1985 ms


### Function for betting

In [33]:
def make_a_bet():
    dev.flush_serial_buffer()  # clear old responses
    
    while True:
        dev.poll_for_response()  # check for responses
        if dev.has_response():
            response = dev.get_next_response()
            bet_keys = [5, 6, 7]  # valid bet keys
            if response['pressed'] and response['key'] in bet_keys:  # ignore releases and invalid keys
                if response['key'] == 5:
                    return 1
                elif response['key'] == 6:
                    return 2
                elif response['key'] == 7:
                    return 3

In [34]:
key = make_a_bet()
print(f"Bet: {key}")

Bet: 3
